### Adaptive algorithms

Some experimental algorithms to compute the derivative of a function iteratively

In [1]:
import numpy as np

In [2]:
def f1(x): # Target function
    return x ** 2 / (x ** 3 + 1.0)

In [3]:
def dev(f, x0, h, scheme): # Finite difference derivative
    match scheme:
        case 'forward':
            return (f(x0 + h) - f(x0)) / h
        case 'backward':
            return (f(x0) - f(x0 - h)) / h
        case 'central':
            return (f(x0 + h) - f(x0 - h)) / (2.0 * h)
        case _:
            return 0.0

In [4]:
def dev_adaptive(f, x0, h0 = 0.1, eps = 1e-06, scheme = 'forward'): # Adaptive finite difference

    h = 2.0 * h0
    j = 0
    err2 = eps + 1.0

    while err2 > eps: # Absolute error criterion

        h /= 2.0
        j += 1

        derivative1 = dev(f, x0, h, scheme)
        derivative2 = dev(f, x0, h / 2.0, scheme)
        diff21 = derivative2 - derivative1
        err2 = np.abs(diff21) if (scheme == 'forward' or scheme == 'backward') else np.abs(diff21) / 3.0 # Richardson error formula applied for derivative2

    rel_err2 = err2 / np.abs(derivative2) if derivative2 != 0.0 else np.nan # Relative error
    
    # Quality factor of the error. Better when the difference is close to zero
    derivative4 = dev(f, x0, h / 4.0, scheme)
    diff42 = derivative4 - derivative2
    ratio = diff21 / diff42 if diff42 != 0.0 else np.inf
    diffQ = np.abs(2.0 - ratio) if (scheme == 'forward' or scheme == 'backward') else np.abs(4.0 - ratio)

    print(f"Loop ended at j = {j}. h = {h}\n")
    return derivative2, err2, rel_err2, diffQ # It returns the derivative, the absolute error and relative error committed, the delta quality factor

In [5]:
x0 = 1.0 # Point target

In [6]:
dev_adaptive( # Forward scheme
    f = f1,
    x0 = x0,
    scheme = 'forward'
)

Loop ended at j = 16. h = 3.0517578125e-06



(0.24999904635478742,
 np.float64(9.536961442790926e-07),
 np.float64(3.8147991289760756e-06),
 np.float64(0.0005341880341882543))

In [7]:
dev_adaptive( # Backward scheme
    f = f1,
    x0 = x0,
    scheme = 'backward'
)

Loop ended at j = 16. h = 3.0517578125e-06



(0.2500009536743164,
 np.float64(9.53677954385057e-07),
 np.float64(3.814697265625e-06),
 np.float64(0.0001907523271782452))

In [8]:
dev_adaptive( # Central scheme
    f = f1,
    x0 = x0,
    scheme = 'central'
)

Loop ended at j = 6. h = 0.003125



(0.2500007629354428,
 np.float64(7.629194313333679e-07),
 np.float64(3.0516684124295055e-06),
 np.float64(7.855757793917562e-05))

In [9]:
dev_adaptive( # Central scheme with smaller epsilon
    f = f1,
    x0 = x0,
    eps = 1e-12,
    scheme = 'central'
)

Loop ended at j = 20. h = 1.9073486328125e-07



(0.24999999994179234, np.float64(0.0), np.float64(0.0), np.float64(4.0))

The quality factor is 0, that means the Richardson error estimation has failed. $\epsilon$ is too small. Let's implement a different adaptive version without $\epsilon$

In [ ]:
def dev_adaptive_Q(f, x0, h0 = 0.1, scheme = 'forward', max_iter = 100):

    p = 1 if scheme in ('forward', 'backward') else 2

    # Initial triple of estimates on steps (h, h/2, h/4)
    h = h0
    d1 = dev(f, x0, h, scheme) # step h
    d2 = dev(f, x0, h / 2.0, scheme) # step h/2
    d4 = dev(f, x0, h / 4.0, scheme) # step h/4

    diff21 = d2 - d1
    diff42 = d4 - d2

    if diff42 == 0.0: # Q undefined -> stop the entire function
        print("h0 is too small. Initial degeneracy d4 == d2")
        return None

    ratio = (d2 - d1) / (d4 - d2)
    diffq_best = np.abs(2.0 ** p - ratio)

    h_best = h / 2.0
    deriv_best = d2
    diff21_best = diff21

    status = 'maximum number of iterations reached'

    for j in range(1, max_iter + 1): # Halving loop reuses the two evaluations that the new window

        d1 = d2 # Old middle becomes new coarse
        d2 = d4 # Old fine becomes new middle
        
        h /= 2.0
        d4 = dev(f, x0, h / 4.0, scheme) # Only one new evaluation

        diff21 = d2 - d1
        diff42 = d4 - d2

        if diff42 == 0.0: # Q undefined -> stop
            status = 'degenerate d4 == d2. Loop stopped'
            break

        ratio = (d2 - d1) / (d4 - d2)
        diffq_new = np.abs(2.0 ** p - ratio)

        if diffq_new < diffq_best:
            diffq_best = diffq_new
            h_best = h / 2.0
            deriv_best = d2
            diff21_best = diff21
        else:
            status = 'converged'
            break

    # Error estimates (Richardson extrapolation on the winning window)
    err_abs = abs(diff21_best) / (2.0 ** p - 1.0)
    rel_err = err_abs / abs(deriv_best) if deriv_best != 0.0 else np.nan

    return {
        'iterations':     j,
        'h':              h_best,
        'derivative':     deriv_best,
        'absolute error': err_abs,
        'relative error': rel_err,
        'diffQ':          diffq_best,
        'status':         status,
    }

In [11]:
dev_adaptive_Q(
    f = f1,
    x0 = x0,
    scheme = 'forward'
)

{'iterations': 13,
 'h': 1.220703125e-05,
 'derivative': 0.2499923706636764,
 'absolute error': 7.6292644735076465e-06,
 'relative error': 3.051798922204536e-05,
 'diffQ': np.float64(3.5763332570937223e-06),
 'status': 'converged'}

In [12]:
dev_adaptive_Q(
    f = f1,
    x0 = x0,
    scheme = 'backward'
)

{'iterations': 14,
 'h': 6.103515625e-06,
 'derivative': 0.2500038147172745,
 'absolute error': 3.814720912487246e-06,
 'relative error': 1.5258650820193507e-05,
 'diffQ': np.float64(4.7683306552137594e-06),
 'status': 'converged'}

In [13]:
dev_adaptive_Q(
    f = f1,
    x0 = x0,
    h0 = 1e-03,
    scheme = 'central'
)

{'iterations': 1,
 'h': 0.0005,
 'derivative': 0.25000007812503444,
 'absolute error': 7.81247474120524e-08,
 'relative error': 3.124988919922628e-07,
 'diffQ': np.float64(8.526522221163901e-06),
 'status': 'converged'}

In [14]:
dev_adaptive_Q(
    f = f1,
    x0 = x0,
    h0 = 1e-06, # Smaller h0. Max accuracy reached for the forward scheme
    scheme = 'forward'
)

{'iterations': 1,
 'h': 5e-07,
 'derivative': 0.24999968761818536,
 'absolute error': 3.126388037344441e-07,
 'relative error': 1.2505567775425582e-06,
 'diffQ': np.float64(0.0),
 'status': 'converged'}

Let's test it on a fast oscillating function

In [15]:
def f2(x): # Hard function
    return np.sqrt(np.abs(np.sin(x ** 3)))

In [16]:
x0 = 2.0 * np.pi # New target point
h0 = 1e-03 # Initial step size

In [17]:
dev_adaptive_Q(
    f = f2,
    x0 = x0,
    h0 = h0,
    scheme = 'forward'
)

{'iterations': 12,
 'h': 2.44140625e-07,
 'derivative': np.float64(-159.58630642376193),
 'absolute error': np.float64(0.008778045753388142),
 'relative error': np.float64(5.500500606912422e-05),
 'diffQ': np.float64(7.809920572965368e-05),
 'status': 'converged'}

In [18]:
dev_adaptive_Q(
    f = f2,
    x0 = x0,
    h0 = h0,
    scheme = 'backward'
)

{'iterations': 12,
 'h': 2.44140625e-07,
 'derivative': np.float64(-159.5687557794463),
 'absolute error': np.float64(0.008772678597779304),
 'relative error': np.float64(5.4977420579156346e-05),
 'diffQ': np.float64(2.412972680221337e-05),
 'status': 'converged'}

In [19]:
dev_adaptive_Q(
    f = f2,
    x0 = x0,
    h0 = h0,
    scheme = 'central'
)

{'iterations': 7,
 'h': 7.8125e-06,
 'derivative': np.float64(-159.578459535485),
 'absolute error': np.float64(0.0009294909215592876),
 'relative error': np.float64(5.824664082263555e-06),
 'diffQ': np.float64(0.0003060375299339668),
 'status': 'converged'}

Let's try to compute the derivative using brute force

In [20]:
def derivative_brute_force(f, x0, min = 2.0, max = 9.0, scheme = 'forward'): # Finds the minimum of diffQ. Then, computes the derivative
    
    # Scheme order p
    p = 1 if scheme in ('forward', 'backward') else 2

    # Array of h values
    hs = 10.0 ** (- np.arange(min, max + 1, 1e-03))

    # Array of diffQs
    diffQs = []

    for h in hs: # Computing diffQs for each h value
    
        derivative1 = dev(f, x0, h, scheme)
        derivative2 = dev(f, x0, h / 2.0, scheme)
        derivative4 = dev(f, x0, h / 4.0, scheme)

        diff21 = derivative2 - derivative1
        diff42 = derivative4 - derivative2

        if diff42 == 0.0:
            diffQs.append(np.nan)
        else:
            ratio = diff21 / diff42
            diffQs.append(np.abs(2.0 ** p - ratio))

    diffQs = np.array(diffQs)

    # Find best values
    min_diffQ = np.nanmin(diffQs[diffQs != 0.0]) # Minimum diffQ found
    min_h = hs[diffQs == min_diffQ] # best value of h found

    # Computing the derivative
    d1 = dev(f, x0, 2.0 * min_h, scheme)
    d2 = dev(f, x0, min_h, scheme)
    err2 = np.abs(d2 - d1) / (2.0 ** p - 1.0) # Richardson error formula
    rel_err2 = err2 / np.abs(d2)

    stats = {
        'h': min_h,
        'diff Q': min_diffQ
    }
    
    return d2, err2, rel_err2, stats

Test on easy function

In [21]:
x0 = 1.0

In [22]:
derivative_brute_force(f1, x0, scheme = 'forward')

(array([0.24998825]),
 array([1.17454023e-05]),
 array([4.69838166e-05]),
 {'h': array([1.87931682e-05]), 'diff Q': np.float64(9.452216787053658e-12)})

In [23]:
derivative_brute_force(f1, x0, scheme = 'backward')

(array([0.25001104]),
 array([1.10380287e-05]),
 array([4.41501656e-05]),
 {'h': array([1.76603782e-05]), 'diff Q': np.float64(2.0116797116997986e-11)})

In [24]:
derivative_brute_force(f1, x0, scheme = 'central')

(array([0.25000039]),
 array([3.8801118e-07]),
 array([1.55204231e-06]),
 {'h': array([0.00111429]), 'diff Q': np.float64(1.5260201990940914e-09)})

Test on hard function (it could fail badly)

In [25]:
x0 = 2.0 * np.pi

In [26]:
derivative_brute_force(f2, x0, scheme = 'forward')

(array([-159.57695102]),
 array([0.00066387]),
 array([4.16018187e-06]),
 {'h': array([1.16144861e-10]), 'diff Q': np.float64(4.3153924877969985e-11)})

In [27]:
derivative_brute_force(f2, x0, scheme = 'backward')

(array([-159.57653099]),
 array([0.00099471]),
 array([6.23345965e-06]),
 {'h': array([2.7733201e-08]), 'diff Q': np.float64(1.1413003875304639e-10)})

In [28]:
derivative_brute_force(f2, x0, scheme = 'central')

(array([-159.57751616]),
 array([2.35613129e-09]),
 array([1.47648074e-11]),
 {'h': array([1.96336028e-09]), 'diff Q': np.float64(1.4568701800499184e-09)})